# 01 · Setting the Stage — Foundations of OpenUSD

> Companion notebook to `docs/stage-setting/` from the Learn OpenUSD curriculum.

## Learning objectives
- Create and manipulate USD stages
- Work with prims (the building blocks of every scene)
- Author attributes and relationships on prims
- Navigate scene hierarchies via prim and property paths
- Choose the right USD file format
- Organize data with metadata
- Handle time-varying data with time codes and time samples

## How to use this notebook
Run cells top-to-bottom. Every example writes into `./_assets/` next to this notebook — delete the folder any time to reset.

In [1]:
import sys
print("Python:", sys.version.split()[0])

from pxr import Usd, Sdf, Gf, UsdGeom, UsdShade
print("pxr import OK — usd-core is installed.")

from pathlib import Path
NB_DIR = Path.cwd()
(NB_DIR / "_assets").mkdir(exist_ok=True)
print("_assets/ ready at:", NB_DIR / "_assets")

Python: 3.12.9


pxr import OK — usd-core is installed.
_assets/ ready at: /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/01_stage_setting/_assets


In [2]:
# Try the repo's helpers; fall back to inline versions if running outside the uv env.
try:
    from lousd.utils.helperfunctions import create_new_stage
    from lousd.utils.visualization import DisplayUSD, DisplayCode
    print("lousd helpers loaded — 3D previews available via DisplayUSD().")
    _HAVE_LOUSD = True
except ImportError:
    import os
    from pathlib import Path as _P
    def create_new_stage(relative_file_path: str):
        from pxr import Usd, Sdf
        ident = os.path.join(os.getcwd(), relative_file_path)
        found = Sdf.Layer.Find(identifier=ident)
        if found:
            return Usd.Stage.Open(found.identifier)
        return Usd.Stage.CreateNew(relative_file_path)
    def DisplayUSD(path, show_usd_code=True):
        from pxr import Usd
        print(f"--- {path} ---")
        print(Usd.Stage.Open(str(path)).ExportToString(addSourceFileComment=False))
    def DisplayCode(path):
        print(f"--- {path} ---")
        print(_P(path).read_text())
    print("Using fallback helpers (lousd not installed).")
    _HAVE_LOUSD = False

lousd helpers loaded — 3D previews available via DisplayUSD().


## 1.1  Stage

A **stage** is the top-level scenegraph you open, author, and save. It's the composed result of one or more USD layers — think of it as the scene or shot you load in a DCC. A single stage might be backed by one file (a robot), or by many files referenced together (a whole factory). Stages are not the files themselves; they're the unified view produced by USD's composition algorithm. Leveraging them well enables non-destructive editing, modularity, and scalability for massive datasets.

**Key terms:** stage, layer, root layer, composition, `Usd.Stage.CreateNew`, `Usd.Stage.Open`, `Usd.Stage.Save`, `CreateInMemory`, `GetRootLayer`, sublayer.

> Source: [docs/stage-setting/stage.md](../docs/stage-setting/stage.md)

### Example 1: Create a USD file and load it as a stage

In [3]:
# Import the `Usd` module from the `pxr` package:
from pxr import Usd

# Define a file path name:
file_path = "_assets/first_stage.usda"
# Create a stage at the given `file_path`:
stage: Usd.Stage = Usd.Stage.CreateNew(file_path)
print(stage.ExportToString(addSourceFileComment=False))

#usda 1.0




In [4]:
DisplayCode("_assets/first_stage.usda")

**What just happened:** `CreateNew` wrote an empty `.usda` file to disk and returned a live stage object bound to that root layer. Because we haven't authored any prims yet, the exported text is effectively empty.

### Example 2: Open and save USD stages

In [5]:
from pxr import Usd

# Open an existing USD stage from disk:
stage: Usd.Stage = Usd.Stage.Open("_assets/first_stage.usda")

# Add a simple prim so we can see a change in the saved file:
stage.DefinePrim("/World", "Xform")

# Save the stage back to disk:
stage.Save()

# Print the stage as text so we can inspect the result:
print(stage.ExportToString(addSourceFileComment=False))

#usda 1.0

def Xform "World"
{
}




In [6]:
DisplayUSD("_assets/first_stage.usda", show_usd_code=True)

**What just happened:** We reopened the layer, defined a `/World` Xform prim on the live stage, and `Save()` flushed the edits to the root layer. Any edits on a stage with no stronger layer target end up in its root layer.

### Example 3: Create a stage in memory

In [7]:
from pxr import Usd

# Create a new stage stored only in memory:
stage: Usd.Stage = Usd.Stage.CreateInMemory()


# Add a prim so the stage contains some data:
stage.DefinePrim("/World", "Xform")

# Print the stage's contents:
print("In-memory stage:")
print(stage.ExportToString(addSourceFileComment=False))

# Export the stage to disk if needed:
stage.Export("_assets/in_memory_stage.usda")

In-memory stage:
#usda 1.0

def Xform "World"
{
}




True

In [8]:
DisplayUSD("_assets/in_memory_stage.usda", show_usd_code=True)

**What just happened:** `CreateInMemory` gave us a stage with no file backing. We still used `DefinePrim` normally, then `Export` materialized the anonymous layer to a concrete `.usda` file on disk. Great for tests and procedural generation.

### Example 4: Working with the root layer

In [9]:
from pxr import Usd, Sdf
import os

# Create a new stage:
stage: Usd.Stage = Usd.Stage.CreateNew("_assets/root_layer_example.usda")

# Get the root layer object:
root_layer: Sdf.Layer = stage.GetRootLayer()
# Use relpath to avoid printing build machine filesystem info.
print("Root layer identifier:", os.path.relpath(root_layer.identifier))

# Add a simple prim so the stage is not empty:
stage.DefinePrim("/World", "Xform")

# Create an additional layer (in a different format) and add it as a sublayer:
extra_layer: Sdf.Layer = Sdf.Layer.CreateNew("_assets/extra_layer.usdc")
# Anchor the path relative to the root layer for better portability
rel_path = "./" + os.path.basename(extra_layer.identifier)
root_layer.subLayerPaths.append(rel_path)

# Save both layers:
stage.Save()
extra_layer.Save()

# Print the contents of the root layer:
print("Root layer contents:")
print(root_layer.ExportToString())

Root layer identifier: _assets/root_layer_example.usda
Root layer contents:
#usda 1.0
(
    subLayers = [
        @./extra_layer.usdc@
    ]
)

def Xform "World"
{
}




In [10]:
DisplayCode("_assets/root_layer_example.usda")

**What just happened:** Every stage owns a root layer (the file you passed to `CreateNew`). We reached it with `GetRootLayer`, added a separate `.usdc` sublayer via `subLayerPaths`, and saved both. Note that sublayers can use different file formats — USDA and USDC freely mix.

## 1.2  Prims

A **prim** (primitive) is the fundamental container in a USD scenegraph. It holds attributes, relationships, and other child prims, forming the hierarchical tree that describes your scene. Each prim has a unique path like `/World/BuildingA/Geometry/building_geo` and an optional type (`Sphere`, `Xform`, `Mesh`, `Scope`, ...) that dictates what data it carries. Generic prims are created with `stage.DefinePrim(path, type)`, but most schema types expose a `Define(stage, path)` convenience that returns a typed wrapper.

**Key terms:** prim, type name, `DefinePrim`, `UsdGeom.Xform`, `UsdGeom.Sphere`, `UsdGeom.Scope`, `UsdGeom.Cube`, `GetChild`, `IsValid`, pseudo-root.

> Source: [docs/stage-setting/prims.md](../docs/stage-setting/prims.md)

### Example 1: Defining a prim

In [11]:
# Import the `Usd` module from the `pxr` package:
from pxr import Usd

# Create a new USD stage with root layer named "prims.usda":
stage: Usd.Stage = create_new_stage("_assets/prims.usda")

# Define a new primitive at the path "/hello" on the current stage:
stage.DefinePrim("/hello")

# Define a new primitive at the path "/world" on the current stage with the prim type, Sphere.
stage.DefinePrim("/world", "Sphere")

stage.Save()

In [12]:
DisplayUSD("_assets/prims.usda", show_usd_code=True)

**What just happened:** `/hello` was defined without a type (an empty container) while `/world` got the `Sphere` type. The exported `.usda` shows `def "hello"` versus `def Sphere "world"` — the type name drives schema behavior.

### Example 2: Defining a sphere prim via the schema API

In [13]:
from pxr import Usd, UsdGeom

file_path = "_assets/sphere_prim.usda"
stage: Usd.Stage = create_new_stage(file_path)

# Define a prim of type `Sphere` at path `/hello`:
sphere: UsdGeom.Sphere = UsdGeom.Sphere.Define(stage, "/hello")
sphere.CreateRadiusAttr().Set(2)

# Save the stage:
stage.Save()

In [14]:
DisplayUSD("_assets/sphere_prim.usda", show_usd_code=True)

**What just happened:** `UsdGeom.Sphere.Define` returned a typed handle with schema-aware methods. `CreateRadiusAttr().Set(2)` authored the radius attribute explicitly — cleaner than raw `DefinePrim` for any prim type with a schema.

### Example 3: Creating a prim hierarchy

In [15]:
from pxr import Usd, UsdGeom

file_path = "_assets/prim_hierarchy.usda"
stage: Usd.Stage = create_new_stage(file_path)

# Define a Scope prim in stage at `/Geometry`
geom_scope: UsdGeom.Scope = UsdGeom.Scope.Define(stage, "/Geometry")
# Define an Xform prim in the stage as a child of /Geometry called GroupTransform
xform: UsdGeom.Xform = UsdGeom.Xform.Define(stage, geom_scope.GetPath().AppendPath("GroupTransform"))
# Define a Cube in the stage as a child of /Geometry/GroupTransform, called Box
cube: UsdGeom.Cube = UsdGeom.Cube.Define(stage, xform.GetPath().AppendPath("Box"))

stage.Save()

In [16]:
DisplayUSD("_assets/prim_hierarchy.usda", show_usd_code=True)

**What just happened:** We used `AppendPath` on each parent's path to build `/Geometry/GroupTransform/Box` — a Scope containing an Xform containing a Cube. Nesting like this is how scenegraphs take shape.

### Example 4: Does the prim exist? (GetChild — missing child)

In [17]:
from pxr import Usd

file_path = "_assets/prim_hierarchy.usda"
stage: Usd.Stage = Usd.Stage.Open(file_path)

prim: Usd.Prim = stage.GetPrimAtPath("/Geometry")
child_prim: Usd.Prim
if child_prim := prim.GetChild("Box"):
    print("Child prim exists")
else:
    print("Child prim DOES NOT exist")

Child prim DOES NOT exist


In [18]:
DisplayCode("_assets/prim_hierarchy.usda")

**What just happened:** `GetChild` only inspects *direct* children. `Box` lives under `GroupTransform`, not directly under `Geometry`, so the lookup returned an invalid prim that evaluates falsy.

### Example 4b: Does the prim exist? (GetChild — real child)

In [19]:
from pxr import Usd

file_path = "_assets/prim_hierarchy.usda"
stage: Usd.Stage = Usd.Stage.Open(file_path)

prim: Usd.Prim = stage.GetPrimAtPath("/Geometry")
child_prim: Usd.Prim
if child_prim := prim.GetChild("GroupTransform"):
    print("Child prim exists")
else:
    print("Child prim DOES NOT exist")

Child prim exists


In [20]:
DisplayCode("_assets/prim_hierarchy.usda")

**What just happened:** `GroupTransform` *is* a direct child of `/Geometry`, so the walrus assignment captured a valid prim and the truthy branch fired.

## 1.3  Attributes

**Attributes** are the typed name-value pairs that carry real data on a prim: a sphere's radius, a cube's size, a mesh's vertices, a material's diffuse color. Each attribute has exactly one data type (defined via `Sdf.ValueTypeNames`), can hold a default value and/or time samples, and is usually authored through the schema-specific API (for example, `sphere.GetRadiusAttr().Set(2.0)`). If no value is explicitly authored, `Get()` resolves to the schema's fallback value.

**Key terms:** attribute, property, schema fallback, value resolution, `GetProperties`, `GetAttributes`, `GetRadiusAttr`, `GetSizeAttr`, `GetDisplayColorAttr`, `GetExtentAttr`, `Set`, `Get`.

> Source: [docs/stage-setting/properties/attributes.md](../docs/stage-setting/properties/attributes.md)

### Example 1: Retrieving properties of a prim

In [21]:
from pxr import Usd, UsdGeom, Gf

file_path = "_assets/attributes_ex1.usda"

stage: Usd.Stage = create_new_stage(file_path)

world_xform: UsdGeom.Xform = UsdGeom.Xform.Define(stage, "/World")

# Define a sphere under the World xForm:
sphere: UsdGeom.Sphere = UsdGeom.Sphere.Define(stage, world_xform.GetPath().AppendPath("Sphere"))

# Define a cube under the World xForm and set it to be 5 units away from the sphere:
cube: UsdGeom.Cube = UsdGeom.Cube.Define(stage, world_xform.GetPath().AppendPath("Cube"))
UsdGeom.XformCommonAPI(cube).SetTranslate(Gf.Vec3d(5, 0, 0))

# Get the property names of the cube prim:
cube_prop_names = cube.GetPrim().GetPropertyNames()

# Print the property names:
for prop_name in cube_prop_names:
    print(prop_name)

stage.Save()

doubleSided
extent
orientation
primvars:displayColor
primvars:displayOpacity
proxyPrim
purpose
size
visibility
xformOp:translate
xformOpOrder


In [22]:
DisplayUSD("_assets/attributes_ex1.usda", show_usd_code=True)

**What just happened:** `GetPropertyNames()` lists every authored and schema-declared property on the cube — you can see transform ops, `size`, `extent`, `displayColor`, and so on. These are both attributes and (potentially) relationships.

### Example 2: Getting attribute values

In [23]:
from pxr import Usd, UsdGeom, Gf

file_path = "_assets/attributes_ex2.usda"
stage: Usd.Stage = create_new_stage(file_path)

world_xform: UsdGeom.Xform = UsdGeom.Xform.Define(stage, "/World")

sphere: UsdGeom.Sphere = UsdGeom.Sphere.Define(stage, world_xform.GetPath().AppendPath("Sphere"))
cube: UsdGeom.Cube = UsdGeom.Cube.Define(stage, world_xform.GetPath().AppendPath("Cube"))
UsdGeom.XformCommonAPI(cube).SetTranslate(Gf.Vec3d(5, 0, 0))

# Get the attributes of the cube prim
cube_attrs = cube.GetPrim().GetAttributes()
for attr in cube_attrs:
    print(attr)

# Get the size, display color, and extent attributes of the cube
cube_size: Usd.Attribute = cube.GetSizeAttr()
cube_displaycolor: Usd.Attribute = cube.GetDisplayColorAttr()
cube_extent: Usd.Attribute = cube.GetExtentAttr()

print(f"Size: {cube_size.Get()}")
print(f"Display Color: {cube_displaycolor.Get()}")
print(f"Extent: {cube_extent.Get()}")

stage.Save()

Usd.Prim(</World/Cube>).GetAttribute('doubleSided')
Usd.Prim(</World/Cube>).GetAttribute('extent')
Usd.Prim(</World/Cube>).GetAttribute('orientation')
Usd.Prim(</World/Cube>).GetAttribute('primvars:displayColor')
Usd.Prim(</World/Cube>).GetAttribute('primvars:displayOpacity')
Usd.Prim(</World/Cube>).GetAttribute('purpose')
Usd.Prim(</World/Cube>).GetAttribute('size')
Usd.Prim(</World/Cube>).GetAttribute('visibility')
Usd.Prim(</World/Cube>).GetAttribute('xformOp:translate')
Usd.Prim(</World/Cube>).GetAttribute('xformOpOrder')
Size: 2.0
Display Color: None
Extent: [(-1, -1, -1), (1, 1, 1)]


In [24]:
DisplayUSD("_assets/attributes_ex2.usda", show_usd_code=True)

**What just happened:** Even though we never authored `size`, `displayColor`, or `extent`, `Get()` returned values — those came from the schema's fallback values via USD's value resolution. The `.usda` file on disk shows nothing for those attributes.

### Example 3: Setting attribute values

In [25]:
from pxr import Usd, UsdGeom, Gf

file_path = "_assets/attributes_ex3.usda"
stage: Usd.Stage = create_new_stage(file_path)

world_xform: UsdGeom.Xform = UsdGeom.Xform.Define(stage, "/World")

sphere: UsdGeom.Sphere = UsdGeom.Sphere.Define(stage, world_xform.GetPath().AppendPath("Sphere"))
cube: UsdGeom.Cube = UsdGeom.Cube.Define(stage, world_xform.GetPath().AppendPath("Cube"))
UsdGeom.XformCommonAPI(cube).SetTranslate(Gf.Vec3d(5,0,0))

# Get the size, display color, and extent attributes of the cube
cube_size: Usd.Attribute = cube.GetSizeAttr()
cube_displaycolor: Usd.Attribute = cube.GetDisplayColorAttr()
cube_extent: Usd.Attribute = cube.GetExtentAttr()

# Modify the size, extent, and display color attributes:
cube_size.Set(cube_size.Get() * 2)
cube_extent.Set(cube_extent.Get() * 2)
cube_displaycolor.Set([(0.0, 1.0, 0.0)])

stage.Save()

In [26]:
DisplayUSD("_assets/attributes_ex3.usda", show_usd_code=True)

**What just happened:** `Set()` authors explicit values, which from now on override the fallbacks and appear in the `.usda`. We doubled the size/extent and painted the cube green.

## 1.4  Relationships

A **relationship** is the second kind of property: instead of a typed value, it stores one or more *paths* that point at other scene objects. Relationships are robust against edits and references — USD translates the targets as paths change — making them ideal for collections, proxy pointers, and material bindings. Author them with `CreateRelationship` / `SetTargets`, and read them back with `GetTargets`.

**Key terms:** relationship, target, path translation, `CreateRelationship`, `SetTargets`, `GetTargets`, `GetRelationship`, `UsdShade.MaterialBindingAPI`, `UsdGeom.Imageable.GetProxyPrimRel`.

> Source: [docs/stage-setting/properties/relationships.md](../docs/stage-setting/properties/relationships.md)

### Example 1: Prim collections with relationships

In [27]:
from pxr import Usd, UsdGeom, Gf

file_path = "_assets/relationships_ex1.usda"
stage = create_new_stage(file_path)

world_xform: UsdGeom.Xform = UsdGeom.Xform.Define(stage, "/World")

# Define a sphere under the World Xform:
sphere: UsdGeom.Sphere = UsdGeom.Sphere.Define(stage, world_xform.GetPath().AppendPath("Sphere"))

# Define a cube under the World Xform and set it to be 5 units away from the sphere:
cube: UsdGeom.Cube = UsdGeom.Cube.Define(stage, world_xform.GetPath().AppendPath("Cube"))
UsdGeom.XformCommonAPI(cube).SetTranslate(Gf.Vec3d(5, 0, 0))

# Create typeless container for the group
group = stage.DefinePrim("/World/Group") 

# Define the relationship
group.CreateRelationship("members", custom=True).SetTargets(
    [sphere.GetPath(), cube.GetPath()]
)

# List relationship targets
members_rel = group.GetRelationship("members")
print("Group members:", [str(p) for p in members_rel.GetTargets()])

stage.Save()

Group members: ['/World/Sphere', '/World/Cube']


In [28]:
DisplayUSD("_assets/relationships_ex1.usda", show_usd_code=True)

**What just happened:** We made a typeless `/World/Group` prim and authored a custom `members` relationship pointing at the Sphere and Cube. The USDA now contains a `rel members = [...]` line — a lightweight way to group prims without moving them in the hierarchy.

### Example 2: Using a built-in relationship (proxyPrim)

In [29]:
from pxr import Usd, UsdGeom

file_path = "_assets/relationships_ex2.usda"
stage = create_new_stage(file_path)

world_xform: UsdGeom.Xform = UsdGeom.Xform.Define(stage, "/World")

# Define a "high cost" Sphere Prim under the World Xform:
high: UsdGeom.Sphere = UsdGeom.Sphere.Define(stage, world_xform.GetPath().AppendPath("HiRes"))

# Define a "low cost" Cube Prim under World Xfrom
low: UsdGeom.Cube = UsdGeom.Cube.Define(stage, world_xform.GetPath().AppendPath("Proxy"))

UsdGeom.Imageable(high).GetPurposeAttr().Set("render")
UsdGeom.Imageable(low).GetPurposeAttr().Set("proxy")

# Author the proxy link on the render Prim
UsdGeom.Imageable(high).GetProxyPrimRel().SetTargets([low.GetPath()])

# Tools that honor proxyPrim should draw the proxy in preview
draw_prim = UsdGeom.Imageable(high).ComputeProxyPrim()  # returns Usd.Prim
print("Preview should draw:", str(draw_prim[0].GetPath() if draw_prim else high.GetPath()))

stage.Save()

Preview should draw: /World/Proxy


In [30]:
DisplayUSD("_assets/relationships_ex2.usda", show_usd_code=True)

**What just happened:** `proxyPrim` is a built-in relationship on `UsdGeom.Imageable`. By tagging `HiRes` as `purpose=render` and `Proxy` as `purpose=proxy`, then pointing `proxyPrim` from render to proxy, a viewer can display the cheap proxy in the viewport while keeping the expensive prim for final rendering.

### Example 3: Material binding relationships

In [31]:
from pxr import Usd, UsdGeom, UsdShade, Gf, Sdf

file_path = "_assets/relationships_ex3.usda"
stage = create_new_stage(file_path)

world_xform: UsdGeom.Xform = UsdGeom.Xform.Define(stage, "/World")


cube_1: UsdGeom.Cube = UsdGeom.Cube.Define(stage, world_xform.GetPath().AppendPath("Cube_1"))
cube_2: UsdGeom.Cube = UsdGeom.Cube.Define(stage, world_xform.GetPath().AppendPath("Cube_2"))
UsdGeom.XformCommonAPI(cube_2).SetTranslate(Gf.Vec3d(5, 0, 0))
cube_3: UsdGeom.Cube = UsdGeom.Cube.Define(stage, world_xform.GetPath().AppendPath("Cube_3"))
UsdGeom.XformCommonAPI(cube_3).SetTranslate(Gf.Vec3d(10, 0, 0))

# Create typeless container for the materials
looks = stage.DefinePrim("/World/Looks")

# Create simple green material for preview
green: UsdShade.Material = UsdShade.Material.Define(stage, looks.GetPath().AppendPath("GreenMat"))
green_ps = UsdShade.Shader.Define(stage, green.GetPath().AppendPath("PreviewSurface"))
green_ps.CreateIdAttr("UsdPreviewSurface")
green_ps.CreateInput("diffuseColor", Sdf.ValueTypeNames.Color3f).Set(Gf.Vec3f(0.0, 1.0, 0.0))
green.CreateSurfaceOutput().ConnectToSource(green_ps.ConnectableAPI(), "surface")

# Create simple red material for preview
red: UsdShade.Material = UsdShade.Material.Define(stage, looks.GetPath().AppendPath("RedMat"))
red_ps = UsdShade.Shader.Define(stage, red.GetPath().AppendPath("PreviewSurface"))
red_ps.CreateIdAttr("UsdPreviewSurface")
red_ps.CreateInput("diffuseColor", Sdf.ValueTypeNames.Color3f).Set(Gf.Vec3f(1.0, 0.0, 0.0))
red.CreateSurfaceOutput().ConnectToSource(red_ps.ConnectableAPI(), "surface")

# Bind materials to Prims
UsdShade.MaterialBindingAPI.Apply(cube_1.GetPrim()).Bind(green)
UsdShade.MaterialBindingAPI.Apply(cube_2.GetPrim()).Bind(green)
UsdShade.MaterialBindingAPI.Apply(cube_3.GetPrim()).Bind(red)

# Verify by reading the direct binding
for prim in [cube_1, cube_2, cube_3]:
    mat = UsdShade.MaterialBindingAPI(prim).GetDirectBinding().GetMaterial()
    print(f"{prim.GetPath()} -> {mat.GetPath() if mat else 'None'}")

stage.Save()

/World/Cube_1 -> /World/Looks/GreenMat
/World/Cube_2 -> /World/Looks/GreenMat
/World/Cube_3 -> /World/Looks/RedMat


In [32]:
DisplayUSD("_assets/relationships_ex3.usda", show_usd_code=True)

**What just happened:** `UsdShade.MaterialBindingAPI.Apply().Bind()` authors a `material:binding` relationship on each cube that targets a `UsdShade.Material`. Two cubes share the green material, one gets red — all via path-valued relationships that survive references and re-parenting.

## 1.5  Time Codes and Time Samples

A **time code** is a unitless point on the stage timeline (think: a frame). A **time sample** is an attribute value authored at a specific time code. Together with `timeCodesPerSecond` stage metadata, they let USD represent animation: authoring a value at time `t` via `attr.Set(value, time=t)` creates a sample, and USD linearly interpolates between samples at playback. Stage-level `startTimeCode` / `endTimeCode` define the playback range.

**Key terms:** time code, time sample, `SetStartTimeCode`, `SetEndTimeCode`, `timeCodesPerSecond`, `Set(value, time=...)`, `GetTimeSamples`, `XformCommonAPI.SetTranslate(..., time=...)`.

> Source: [docs/stage-setting/timecodes-timesamples.md](../docs/stage-setting/timecodes-timesamples.md)

### Setup: create the sample stage

In [33]:
# Import the necessary modules from the `pxr` library:
from pxr import Usd, UsdGeom, Gf

# Create a new USD stage file named "timecode_sample.usda":
file_path = "_assets/timecode_sample.usda"
stage: Usd.Stage = create_new_stage(file_path)

# Define a transform ("Xform") primitive at the "/World" path:
world: UsdGeom.Xform = UsdGeom.Xform.Define(stage, "/World")

# Define a Sphere primitive as a child of the transform at "/World/Sphere" path:
sphere: UsdGeom.Sphere = UsdGeom.Sphere.Define(stage, world.GetPath().AppendPath("Sphere"))

# Define a blue Cube as a background prim:
box: UsdGeom.Cube = UsdGeom.Cube.Define(stage, world.GetPath().AppendPath("Backdrop"))
box.GetDisplayColorAttr().Set([(0.0, 0.0, 1.0)])
cube_xform_api = UsdGeom.XformCommonAPI(box)
cube_xform_api.SetScale(Gf.Vec3f(5, 5, 0.1))
cube_xform_api.SetTranslate(Gf.Vec3d(0, 0, -2))

# Save the stage to the file:
stage.Save()

In [34]:
DisplayUSD("_assets/timecode_sample.usda", show_usd_code=True)

### Example 1: Setting start and end time codes

In [35]:
from pxr import Usd

# Open stage from starting point usda
stage: Usd.Stage = Usd.Stage.Open("_assets/timecode_sample.usda")

# Set the `start` and `end` time codes for the stage:
stage.SetStartTimeCode(1)
stage.SetEndTimeCode(60)

# Export to a new flattened layer for this example.
stage.Export("_assets/timecode_ex1.usda", addSourceFileComment=False)

True

In [36]:
DisplayUSD("_assets/timecode_ex1.usda", show_usd_code=True)

**What just happened:** `SetStartTimeCode` / `SetEndTimeCode` author stage-level metadata (`startTimeCode = 1`, `endTimeCode = 60`) at the top of the layer — the playback range for any downstream consumer.

### Example 2a: Setting translation time samples

In [37]:
from pxr import Usd, UsdGeom, Gf

# Open stage from example 1
stage: Usd.Stage = Usd.Stage.Open("_assets/timecode_ex1.usda")

sphere: UsdGeom.Sphere = UsdGeom.Sphere.Get(stage, "/World/Sphere")

# Clear any existing translation
if translate_attr := sphere.GetTranslateOp().GetAttr():
    translate_attr.Clear()

# Create XformCommonAPI object for the sphere
sphere_xform_api = UsdGeom.XformCommonAPI(sphere)

# Set translation of the sphere at time 1
sphere_xform_api.SetTranslate(Gf.Vec3d(0,  5.50, 0), time=1)
# Set translation of the sphere at time 30
sphere_xform_api.SetTranslate(Gf.Vec3d(0, -4.50, 0), time=30)
# Set translation of the sphere at time 45
sphere_xform_api.SetTranslate(Gf.Vec3d(0, -5.00, 0), time=45)
# Set translation of the sphere at time 50
sphere_xform_api.SetTranslate(Gf.Vec3d(0, -3.25, 0), time=50)
# Set translation of the sphere at time 60
sphere_xform_api.SetTranslate(Gf.Vec3d(0,  5.50, 0), time=60)

# Export to a new flattened layer for this example.
stage.Export("_assets/timecode_ex2a.usda", addSourceFileComment=False)

True

In [38]:
DisplayUSD("_assets/timecode_ex2a.usda", show_usd_code=True)

**What just happened:** Passing `time=...` to `SetTranslate` authors time samples on the sphere's translate op. The exported USDA shows a `.timeSamples = { 1: ..., 30: ..., 45: ..., 50: ..., 60: ... }` block — USD linearly interpolates between them at playback.

### Example 2b: Setting scale time samples

In [39]:
from pxr import Usd, UsdGeom, Gf

# Open stage from example 2a
stage: Usd.Stage = Usd.Stage.Open("_assets/timecode_ex2a.usda")

sphere: UsdGeom.Sphere = UsdGeom.Sphere.Get(stage, "/World/Sphere")

if scale_attr := sphere.GetScaleOp().GetAttr():
    scale_attr.Clear()

sphere_xform_api = UsdGeom.XformCommonAPI(sphere)

# Set scale of the sphere at time 1
sphere_xform_api.SetScale(Gf.Vec3f(1.00, 1.00, 1.00), time=1)  
# Set scale of the sphere at time 30
sphere_xform_api.SetScale(Gf.Vec3f(1.00, 1.00, 1.00), time=30)   
# Set scale of the sphere at time 45
sphere_xform_api.SetScale(Gf.Vec3f(1.00, 0.20, 1.25), time=45)   
# Set scale of the sphere at time 50
sphere_xform_api.SetScale(Gf.Vec3f(0.75, 2.00, 0.75), time=50)  
# Set scale of the sphere at time 60
sphere_xform_api.SetScale(Gf.Vec3f(1.00, 1.00, 1.00), time=60)  

# Export to a new flattened layer for this example.
stage.Export("_assets/timecode_ex2b.usda", addSourceFileComment=False)

True

In [40]:
DisplayUSD("_assets/timecode_ex2b.usda", show_usd_code=True)

**What just happened:** The same time-sample technique works on any animatable attribute. Now both translate *and* scale are animated — the sphere bounces and squashes.

## 1.6  Prim and Property Paths

A **path** is the unique address of a prim or property within a stage, expressed as a forward-slash string (`/World/Geometry/Box`) and modeled by `Sdf.Path`. Prims use slashes; properties use a trailing dot (`/Geo/Sphere.radius`); variants use curly braces (`/Geo/Sphere{shape=round}`). `Sdf.Path` supports composition operations like `AppendChild`, `AppendProperty`, `GetParentPath`, and predicates like `IsPrimPath`. Paths are how you author, query, and navigate everything in USD.

**Key terms:** `Sdf.Path`, `GetPrimAtPath`, `GetPropertyAtPath`, `AppendChild`, `AppendProperty`, `IsPrimPath`, `GetParentPath`, `IsValid`, pseudo-root.

> Source: [docs/stage-setting/prim-property-paths.md](../docs/stage-setting/prim-property-paths.md)

### Example 1: Getting, validating, and defining prims at path

In [41]:
from pxr import Usd

stage: Usd.Stage = create_new_stage("_assets/paths.usda")
stage.DefinePrim("/hello")
stage.DefinePrim("/hello/world")

# Get the primitive at the path "/hello" from the current stage
hello_prim: Usd.Prim = stage.GetPrimAtPath("/hello")

# Get the primitive at the path "/hello/world" from the current stage
hello_world_prim: Usd.Prim = stage.GetPrimAtPath("/hello/world")

# Get the primitive at the path "/world" from the current stage
# Note: This will return an invalid prim because "/world" does not exist, but if changed to "/hello/world" it will return a valid prim
world_prim: Usd.Prim = stage.GetPrimAtPath("/world")

# Print whether the primitive is valid
print("Is /hello a valid prim? ", hello_prim.IsValid())
print("Is /hello/world a valid prim? ", hello_world_prim.IsValid())
print("Is /world a valid prim? ", world_prim.IsValid())

stage.Save()

Is /hello a valid prim?  True
Is /hello/world a valid prim?  True
Is /world a valid prim?  False


In [42]:
DisplayCode("_assets/paths.usda")

**What just happened:** `GetPrimAtPath` always returns *some* prim object — but calling `IsValid()` tells you whether it actually resolves to an existing location on the stage. Always guard before using.

### Example 2: Build and navigate prim paths with Sdf.Path

In [43]:
from pxr import Usd, UsdGeom, Sdf

stage = create_new_stage("_assets/paths_build_and_nav.usda")

# Build prim paths via Sdf.Path
world_path = Sdf.Path("/World")
geometry_path = world_path.AppendChild("Geometry")  # /World/Geometry
sphere_path = geometry_path.AppendChild("Sphere")  # /World/Geometry/Sphere

looks_path = world_path.AppendChild("Looks")  # /World/Looks
material_path = looks_path.AppendChild("Material")  # /World/Looks/Material

# Define prims at those paths
stage.DefinePrim(world_path)
stage.DefinePrim(geometry_path)
UsdGeom.Sphere.Define(stage, sphere_path)
stage.DefinePrim(looks_path)
stage.DefinePrim(material_path)

# Path checks and basic navigation
print("sphere_path IsPrimPath:", sphere_path.IsPrimPath())
print("sphere_path parent:",    sphere_path.GetParentPath())
print("Geometry prim valid:",   stage.GetPrimAtPath(geometry_path).IsValid())
print("\nmaterial_path IsPrimPath:", material_path.IsPrimPath())
print("material_path parent:",    material_path.GetParentPath())
print("Looks prim valid:",   stage.GetPrimAtPath(looks_path).IsValid())

stage.Save()


sphere_path IsPrimPath: True
sphere_path parent: /World/Geometry
Geometry prim valid: True

material_path IsPrimPath: True
material_path parent: /World/Looks
Looks prim valid: True


In [44]:
DisplayCode("_assets/paths_build_and_nav.usda")

**What just happened:** Instead of concatenating strings, we built `Sdf.Path` objects with `AppendChild` and used them directly in `DefinePrim` / `UsdGeom.Sphere.Define`. `IsPrimPath` and `GetParentPath` show how to introspect and walk the hierarchy.

### Example 3: Author an attribute and a relationship from property paths

In [45]:
from pxr import Usd, UsdGeom, Sdf

stage = create_new_stage("_assets/paths_property_authoring.usda")

# A prim to work with
sphere = UsdGeom.Sphere.Define(stage, "/World/Geom/Sphere")

# Create a property path for the attribute /World/Geom/Sphere.userProperties:tag
attr_property_path = sphere.GetPath().AppendProperty("userProperties:tag")

# Working with the property path for the attribute
owner_prim = stage.GetPrimAtPath(attr_property_path.GetPrimPath())
attr_name = stage.GetPropertyAtPath(attr_property_path).GetPath().name  # "userProperties:tag"
print(f"Attribute property '{attr_name}' has been defined on {owner_prim.GetPath()} after AppendProperty: {owner_prim.GetAttribute(attr_name).IsDefined()}")

# Define the attribute on the owner prim
attr = owner_prim.CreateAttribute(attr_name, Sdf.ValueTypeNames.String)
print(f"\nAttribute property '{attr_name}' has been defined on {owner_prim.GetPath()} after CreateAttribute: {owner_prim.GetAttribute(attr_name).IsDefined()}")

attr.Set("surveyed")
print(f"Attribute value after Set: {stage.GetAttributeAtPath(attr_property_path).Get()}")

# Create a relationship from a property path
marker = UsdGeom.Xform.Define(stage, "/World/Markers/MarkerA")
# Create a property path for the relationship
rel_property_path = sphere.GetPath().AppendProperty("my:ref")  # /World/Geom/Sphere.my:ref

# Working with the property path for the relationship
owner_prim = stage.GetPrimAtPath(rel_property_path.GetPrimPath())
rel_name = stage.GetPropertyAtPath(rel_property_path).GetPath().name  # "my:ref"
print(f"\nRelationship property '{rel_name}' has been defined on {owner_prim.GetPath()} after AppendProperty: {owner_prim.GetRelationship(rel_name).IsDefined()}")

# Define the relationship on the owner prim
rel = owner_prim.CreateRelationship(rel_name)
print(f"\nRelationship property '{rel_name}' has been defined on {owner_prim.GetPath()} after CreateRelationship: {owner_prim.GetRelationship(rel_name).IsDefined()}")

rel.AddTarget(marker.GetPath())
print(f"Relationship targets after AddTarget: {[str(p) for p in stage.GetRelationshipAtPath(rel_property_path).GetTargets()]}")

stage.Save()


Attribute property 'userProperties:tag' has been defined on /World/Geom/Sphere after AppendProperty: False

Attribute property 'userProperties:tag' has been defined on /World/Geom/Sphere after CreateAttribute: True
Attribute value after Set: surveyed

Relationship property 'my:ref' has been defined on /World/Geom/Sphere after AppendProperty: False

Relationship property 'my:ref' has been defined on /World/Geom/Sphere after CreateRelationship: True
Relationship targets after AddTarget: ['/World/Markers/MarkerA']


In [46]:
DisplayCode("_assets/paths_property_authoring.usda")

**What just happened:** `AppendProperty` builds a *property* path (`/World/Geom/Sphere.userProperties:tag`) but does not author anything — you still need `CreateAttribute` / `CreateRelationship`. Once authored, you can round-trip through `GetAttributeAtPath` and `GetRelationshipAtPath`.

## 1.7  USD File Formats

USD ships four native file formats: **USDA** (human-readable ASCII, great for small/interface layers), **USDC** (compressed binary crate, great for data-heavy layers), **USD** (auto-detects ASCII or binary; format can swap without breaking references), and **USDZ** (atomic zipped archive for shipping complete assets, optimal for XR). The production guidance is: prefer crate binary for big data, use text for small aggregator layers, and package with USDZ.

**Key terms:** `.usda`, `.usdc`, `.usd`, `.usdz`, crate binary, layer, `Export`.

> Source: [docs/stage-setting/usd-file-formats.md](../docs/stage-setting/usd-file-formats.md)

The source lesson has no executable examples, so the cells below are a defensive demo: author one stage and export it to each format, then compare sizes.

### Example: Export the same stage to .usda / .usdc / .usdz and compare sizes

In [47]:
from pathlib import Path
from pxr import Usd, UsdGeom, UsdUtils

# Build a small stage in memory once, then write it to each format for size comparison.
stage = Usd.Stage.CreateInMemory()
UsdGeom.Xform.Define(stage, "/World")
UsdGeom.Sphere.Define(stage, "/World/Sphere")

usda_path = "_assets/formats_demo.usda"
usdc_path = "_assets/formats_demo.usdc"
usdz_path = "_assets/formats_demo.usdz"

# .usda and .usdc can be written with stage.Export directly.
stage.Export(usda_path)
stage.Export(usdc_path)

# .usdz is a package layer — it cannot be written via stage.Export.
# Use UsdUtils.CreateNewUsdzPackage to wrap an existing .usda into a .usdz archive.
try:
    UsdUtils.CreateNewUsdzPackage(usda_path, usdz_path)
except Exception as e:
    print(f"USDZ packaging failed ({type(e).__name__}): {e}")

for t in [usda_path, usdc_path, usdz_path]:
    p = Path(t)
    if p.exists():
        print(f"{t:40s}  {p.stat().st_size:>8d} bytes")
    else:
        print(f"{t:40s}  (not created)")


_assets/formats_demo.usda                      136 bytes
_assets/formats_demo.usdc                      700 bytes
_assets/formats_demo.usdz                      302 bytes


In [48]:
# Only .usda is directly readable as text; .usdc and .usdz are binary.
DisplayCode("_assets/formats_demo.usda")

**What just happened:** One stage, three formats, three file sizes. USDA is bigger because it's text, USDC squeezes the same data into a compact binary layout, and USDZ wraps everything into an uncompressed zip archive. In production you'd pick based on role: interface layers get `.usda`, heavy geometry gets `.usdc`, shippable assets get `.usdz`.

## 1.8  USD Modules

OpenUSD's Python API is split across many modules under the `pxr` namespace. The three you will use constantly are: **`Usd`** (core stages, prims, properties, composition), **`Sdf`** (scene description foundation — layers, paths, value types), and **`Gf`** (graphics foundation — linear algebra types like `Vec3d`, `Matrix4d`, `Quatf`). On top of those sit schema modules like `UsdGeom`, `UsdShade`, and `UsdPhysics` that provide typed APIs for specific domains.

**Key terms:** `pxr`, `Usd`, `Sdf`, `Gf`, `UsdGeom`, `UsdShade`, `UsdPhysics`, schema module, base package.

> Source: [docs/stage-setting/usd-modules.md](../docs/stage-setting/usd-modules.md)

The source lesson has no executable examples, so here is a defensive demo that imports each core module and calls one function from it.

### Example: Touring Usd, Sdf, and Gf

In [49]:
from pxr import Usd, Sdf, Gf

# Module paths on disk and a short summary of each:
print("Usd module file:", Usd.__file__)
print("Sdf module file:", Sdf.__file__)
print("Gf  module file:", Gf.__file__)
print()

# Usd: core client API. Create an anonymous stage and author a prim.
stage = Usd.Stage.CreateInMemory()
stage.DefinePrim("/World", "Xform")
print("Usd.Stage.CreateInMemory prim count:", len(list(stage.Traverse())))

# Sdf: layers, paths, value types. Build and inspect a path.
path = Sdf.Path("/Foo/Bar")
print("Sdf.Path parent of /Foo/Bar:", path.GetParentPath())
print("Sdf.ValueTypeNames.Float3:", Sdf.ValueTypeNames.Float3)

# Gf: graphics math primitives.
v = Gf.Vec3d(1, 2, 3) + Gf.Vec3d(4, 5, 6)
print("Gf.Vec3d(1,2,3) + Gf.Vec3d(4,5,6) =", v)
print("Gf.Vec3d length:", v.GetLength())

Usd module file: /Users/stranzersweb/Projects/LearnOpenUSD/.venv/lib/python3.12/site-packages/pxr/Usd/__init__.py
Sdf module file: /Users/stranzersweb/Projects/LearnOpenUSD/.venv/lib/python3.12/site-packages/pxr/Sdf/__init__.py
Gf  module file: /Users/stranzersweb/Projects/LearnOpenUSD/.venv/lib/python3.12/site-packages/pxr/Gf/__init__.py

Usd.Stage.CreateInMemory prim count: 1
Sdf.Path parent of /Foo/Bar: /Foo
Sdf.ValueTypeNames.Float3: float3
Gf.Vec3d(1,2,3) + Gf.Vec3d(4,5,6) = (5, 7, 9)
Gf.Vec3d length: 12.449899597988733


In [50]:
# No layer on disk for this one — export the in-memory stage so we can inspect it.
from pxr import Usd as _Usd
_s = _Usd.Stage.CreateInMemory()
_s.DefinePrim("/World", "Xform")
_s.Export("_assets/modules_demo.usda")
DisplayCode("_assets/modules_demo.usda")

**What just happened:** We imported the three foundational modules, printed where each lives on disk, and called one representative function from each: `Usd.Stage.CreateInMemory` (stage lifecycle), `Sdf.Path.GetParentPath` (path manipulation), and `Gf.Vec3d` arithmetic (math primitives).

## 1.9  Metadata

**Metadata** is supplementary, non-animatable name-value data attached to prims, properties, or layers. It's separate from the schema's typed attributes, lives in a dictionary, and is meant for things like author info, timestamps, notes, or render hints. The two conventional keys are `customData` (free-form user data) and `assetInfo` (asset pipeline metadata). Access it with `GetMetadata` / `SetMetadata` on any `Usd.Object`, or at the stage level with `stage.GetMetadata` / `stage.SetMetadata`.

**Key terms:** metadata, `customData`, `assetInfo`, `GetMetadata`, `SetMetadata`, `SetAssetInfo`, `GetMetadataByDictKey`, stage `comment`.

> Source: [docs/stage-setting/metadata.md](../docs/stage-setting/metadata.md)

The source lesson has no executable examples, so here is a defensive demo that authors and reads back both prim-level and stage-level metadata.

### Example: Author and read metadata

In [51]:
from pxr import Usd, UsdGeom

file_path = "_assets/metadata_demo.usda"
stage: Usd.Stage = create_new_stage(file_path)

# Stage-level metadata: a free-form comment on the root layer.
stage.SetMetadata("comment", "Authored by the Learn OpenUSD notebook on 2026-04-11.")

# Create a prim to attach metadata to.
sphere = UsdGeom.Sphere.Define(stage, "/World/Sphere")
prim = sphere.GetPrim()

# customData: arbitrary user data in a dict.
prim.SetMetadata(
    "customData",
    {"author": "learner", "created": "2026-04-11", "notes": "demo sphere"},
)

# assetInfo: structured asset-pipeline metadata.
prim.SetAssetInfo({"name": "DemoSphere", "version": "1.0"})

# Read metadata back.
print("stage comment:", stage.GetMetadata("comment"))
print("prim customData:", prim.GetMetadata("customData"))
print("prim assetInfo:", prim.GetAssetInfo())

# GetMetadataByDictKey is the efficient way to read a single key out of a dict metadata.
print("customData.author:", prim.GetMetadataByDictKey("customData", "author"))

stage.Save()

stage comment: Authored by the Learn OpenUSD notebook on 2026-04-11.
prim customData: {'author': 'learner', 'created': '2026-04-11', 'notes': 'demo sphere', 'userDocBrief': 'Defines a primitive sphere centered at the origin.'}
prim assetInfo: {'name': 'DemoSphere', 'version': '1.0'}
customData.author: learner


In [52]:
DisplayUSD("_assets/metadata_demo.usda", show_usd_code=True)

**What just happened:** We set a stage-level `comment`, then authored both `customData` and `assetInfo` dictionaries on the prim. The exported USDA shows `customData = { ... }` and `assetInfo = { ... }` blocks above the prim definition — metadata lives alongside the schema data without mixing with it.

## Key Takeaways

- A **stage** is the composed scenegraph built from one or more layers; it enables non-destructive editing, layering, and referencing for complex, multi-collaborator projects.
- **Prims** are the fundamental containers. They hold attributes, relationships, and child prims, and their type name drives schema behavior — prefer schema `Define` over raw `DefinePrim` when a schema exists.
- **Attributes** carry typed name-value data, support fallback values via value resolution, and can be animated with time samples. Use schema-specific getters/setters where available.
- **Relationships** store path targets that survive edits and references — ideal for material binding, collections, proxy prims, and any non-hierarchical link.
- **Paths** (`Sdf.Path`) uniquely identify every prim and property. Build them with `AppendChild` / `AppendProperty` and always check `IsValid` before using the result of `GetPrimAtPath`.
- **Time codes and time samples** drive animation via stage-level `startTimeCode`/`endTimeCode` and per-attribute `Set(value, time=...)` with linear interpolation between samples.
- **File formats**, **modules**, and **metadata** round out the foundation: choose `.usda`/`.usdc`/`.usd`/`.usdz` by role; rely on `Usd`/`Sdf`/`Gf` plus schema modules; use `customData` and `assetInfo` for supplementary dictionary data.

## Next up

Continue to **`02_scene_description_blueprints.ipynb`** to learn how USD's scene description layer (`Sdf`) works under the hood and how to build reusable blueprints for bigger scenes.